In [ ]:
import time
import requests
from datetime import datetime
from itertools import chain
import pandas as pd
from bs4 import BeautifulSoup

In [56]:
def get_pages(num_pages: int):
    for i in range(num_pages):
        if i % 10 == 0:
            print("Getting page", i, "/", num_pages)

        try:
            response = requests.get(
                url=f"https://www.ufc.com/trending/all?page={i}", 
                # Pretending to request the page from a Chrome browser so it doesn't block the crawler
                headers={
                "User-Agent": "Mozilla/5.0 (Windows NT 11.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/134.0.6998.166 Safari/537.36"
            })
            yield response.text
        except Exception as e:
            print("Error getting page", i)
            print(e)

            yield None

In [58]:
# 200 pages is enough for about 8 months of articles
num_pages_to_get = 200
pages = [
    page for page in get_pages(num_pages_to_get)
    if page is not None
]
print("Got", len(pages), "pages")

Getting page 0 / 200
Getting page 10 / 200
Getting page 20 / 200
Getting page 30 / 200
Getting page 40 / 200
Getting page 50 / 200
Getting page 60 / 200
Getting page 70 / 200
Getting page 80 / 200
Getting page 90 / 200
Getting page 100 / 200
Getting page 110 / 200
Getting page 120 / 200
Getting page 130 / 200
Getting page 140 / 200
Getting page 150 / 200
Getting page 160 / 200
Getting page 170 / 200
Getting page 180 / 200
Getting page 190 / 200
Got 200 pages


In [59]:
def get_articles_data(page_content: str):
    soup = BeautifulSoup(page_content, "html.parser")
    articles = soup.find_all("a", "c-card--grid-card-trending")

    articles_data = [
        {
            "type": link.find_all("span", "c-card--grid-card-trending__info-prefix")[0].text,
            "time_ago": link.find_all("div", "field--name-datetime")[0].text,
            "headline": link.find_all("h3", "c-card--grid-card-trending__headline")[0].text.strip(),
            "link": link["href"]
        }
        for link in articles
    ]

    for data in articles_data:
        yield data

In [60]:
flattened_articles = list(chain.from_iterable(get_articles_data(page) for page in pages))
flattened_articles

[{'type': 'Interviews',
  'time_ago': '1 hour ago',
  'headline': 'Ciryl Gane Pre-Fight Interview | UFC 321',
  'link': '/video/150984'},
 {'type': 'Interviews',
  'time_ago': '1 hour ago',
  'headline': 'Tom Aspinall Pre-Fight Interview | UFC 321',
  'link': '/video/150983'},
 {'type': "Dana White's Contender Series",
  'time_ago': '3 hours ago',
  'headline': "Week 6 Preview | Dana White's Contender Series…",
  'link': '/news/week-6-preview-dana-whites-contender-series-season-9'},
 {'type': 'Announcements',
  'time_ago': '20 hours ago',
  'headline': 'Updates To UFC Vancouver',
  'link': '/news/updates-ufc-vancouver'},
 {'type': 'Highlights',
  'time_ago': '20 hours ago',
  'headline': 'Greatest Finishes | Noche UFC: Silva vs Lopes',
  'link': '/video/150976'},
 {'type': 'Fight Coverage',
  'time_ago': '20 hours ago',
  'headline': 'The Scorecard | Noche UFC',
  'link': '/news/scorecard-noche-ufc-san-antonio'},
 {'type': 'Athletes',
  'time_ago': '21 hours ago',
  'headline': 'KYLE D

In [ ]:
articles_df = pd.DataFrame(data=flattened_articles).set_index("link")
articles_df

,type,time_ago,headline
link,,,
/video/150984,Interviews,1 hour ago,Ciryl Gane Pre-Fight Interview | UFC 321
/video/150983,Interviews,1 hour ago,Tom Aspinall Pre-Fight Interview | UFC 321
/news/week-6-preview-dana-whites-contender-series-season-9,Dana White's Contender Series,3 hours ago,Week 6 Preview | Dana White's Contender Series…
/news/updates-ufc-vancouver,Announcements,20 hours ago,Updates To UFC Vancouver
/video/150976,Highlights,20 hours ago,Greatest Finishes | Noche UFC: Silva vs Lopes
...,...,...,...
/video/144850,Interviews,8 months ago,Punahele Soriano Octagon Interview | UFC Fight...
/video/144847,Interviews,8 months ago,Jacobe Smith Post-Fight Interview | UFC Fight ...
/video/144845,Highlights,8 months ago,Felipe Bunes Snatches First-Round Submission |...


In [68]:
filename = f"./article_links_{datetime.now().strftime('%d-%m-%y_%H-%M-%S')}.csv"
articles_df.to_csv(filename)